# Human vs. Machine Translation Detection Codebase

This Jupyter Notebook focuses on preparing datasets for detecting human versus machine-generated translations. It leverages the FLORES+ dataset from Hugging Face to create parallel corpora and monolingual text files for English, Swahili, and Afrikaans.

## Key Steps:
- **Authentication**: Logs into Hugging Face Hub using a token.
- **Data Loading**: Downloads development and devtest splits for English (eng_Latn), Swahili (swh_Latn), and Afrikaans (afr_Latn) from the FLORES+ dataset.
- **Data Processing**: Converts datasets to Pandas Series, concatenates splits, and creates parallel DataFrames for English-Swahili and English-Afrikaans pairs.
- **Export**: Saves parallel corpora as CSV files and monolingual texts as TXT files.
- **Validation**: Prints dataset sizes and sample entries for verification.

The notebook ensures data integrity by checking lengths and handling text formatting (e.g., replacing newlines). Output files include `flores_plus_eng_swh_parallel.csv`, `flores_plus_eng_afr_parallel.csv`, `english.txt`, and `swahili.txt`. This setup supports downstream tasks in translation detection research.

In [ ]:
from huggingface_hub import login

# Login to Hugging Face Hub (replace with your token)
login(HF_TOKEN)

In [ ]:
from datasets import load_dataset
import pandas as pd

DS = "openlanguagedata/flores_plus"

# Load English
eng_dev = load_dataset(DS, "eng_Latn", split="dev")
eng_devtest = load_dataset(DS, "eng_Latn", split="devtest")

# Load Swahili
swh_dev = load_dataset(DS, "swh_Latn", split="dev")
swh_devtest = load_dataset(DS, "swh_Latn", split="devtest")

# Load Afrikaans
afr_dev = load_dataset(DS, "afr_Latn", split="dev")
afr_devtest = load_dataset(DS, "afr_Latn", split="devtest")

print("ENG dev/devtest:", len(eng_dev), len(eng_devtest))
print("SWH dev/devtest:", len(swh_dev), len(swh_devtest))
print("AF dev/devtest:", len(afr_dev), len(afr_devtest))

In [ ]:
def to_series(ds, col_guess=("sentence", "text")):
    cols = set(ds.column_names)
    for c in col_guess:
        if c in cols:
            return pd.Series(ds[c])
    raise ValueError(f"Could not find a text column. Available columns: {ds.column_names}")

eng = pd.concat([to_series(eng_dev), to_series(eng_devtest)], ignore_index=True)
swh = pd.concat([to_series(swh_dev), to_series(swh_devtest)], ignore_index=True)
afr = pd.concat([to_series(afr_dev), to_series(afr_devtest)], ignore_index=True)


if len(eng) != len(swh):
    raise ValueError(f"Length mismatch: eng={len(eng)} vs swh={len(swh)}")

df_parallel = pd.DataFrame({"english": eng, "swahili": swh})
df_parallel.head(), len(df_parallel)


In [ ]:
out_path = "flores_plus_eng_swh_parallel.csv"
df_parallel.to_csv(out_path, index=False)
print("Saved:", out_path)


In [ ]:
df_parallel = pd.DataFrame({"english": eng, "afrikaans": afr})
df_parallel.head(), len(df_parallel)

In [ ]:
out_path = "flores_plus_eng_afr_parallel.csv"
df_parallel.to_csv(out_path, index=False)
print("Saved:", out_path)

In [ ]:
texts = list(eng_dev["text"]) + list(eng_devtest["text"])
out_path = "english.txt"
with open(out_path, "w", encoding="utf-8") as f:
    for t in texts:
        f.write(t.replace("\n", " ") + "\n")
print(len(texts))

In [ ]:
texts = list(swh_dev["text"]) + list(swh_devtest["text"])
out_path = "swahili.txt"
with open(out_path, "w", encoding="utf-8") as f:
    for t in texts:
        f.write(t.replace("\n", " ") + "\n")
print(len(texts))

In [ ]:
for i in [0, 1, 2, 500, 1000, 2000]:
    if i < len(df_parallel):
        print("\n---", i, "---")
        print("EN:", df_parallel.loc[i, "english"])
        print("SW:", df_parallel.loc[i, "swahili"])
